# 📊 Regresión Logística Binaria — Predicción de Calidad Alta del Vino

Construcción, entrenamiento y evaluación de un modelo de **Regresión Logística Binaria** para predecir
si un vino tinto es de **alta calidad** (`es_alta_calidad = 1`, quality ≥ 7).

| | |
|---|---|
| **Variable objetivo (Y)** | `es_alta_calidad` (1 = quality ≥ 7, 0 = quality < 7) |
| **Variables predictoras (X)** | `alcohol`, `volatile acidity`, `sulphates`, `citric acid`, `fixed acidity`, `density`, `pH`, `chlorides`, `residual sugar`, `free sulfur dioxide`, `total sulfur dioxide` |
| **Modelo** | `LogisticRegression` (scikit-learn) |
| **Estrategia de validación** | Tres splits: 80/20 · 60/40 · 70/30 |

> **Dataset:** UCI Wine Quality — 1,599 muestras de vino tinto con propiedades fisicoquímicas y calificación sensorial.

## I. Configuración Inicial — Importaciones y Estilo Visual

In [ ]:
# ── CELDA 1: Importaciones ────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve,
    confusion_matrix, ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')

# Estilo visual consistente con el proyecto
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

print("✅ Librerías importadas correctamente")
print(f"   pandas   {pd.__version__}")
print(f"   numpy    {np.__version__}")
print(f"   seaborn  {sns.__version__}")

## II. Carga de Datos desde UCI

Cargamos el dataset `winequality-red.csv` y generamos directamente la variable objetivo binaria
`es_alta_calidad` a partir de la columna `quality`.

**Criterio de binarización:**
- `quality >= 7` → `es_alta_calidad = 1` (vino de alta calidad)
- `quality < 7`  → `es_alta_calidad = 0` (vino de calidad estándar o baja)

In [ ]:
# ── CELDA 2: Carga de datos desde UCI ────────────────────────────────────────
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"

df_vino = pd.read_csv(URL, sep=";")

# Crear variable objetivo binaria: quality >= 7 → alta calidad
df_vino['es_alta_calidad'] = (df_vino['quality'] >= 7).astype(int)

print(f"✅ Datos cargados: {df_vino.shape[0]:,} filas × {df_vino.shape[1]} columnas")
print(f"\nColumnas disponibles:\n{df_vino.columns.tolist()}")
print(f"\nBalance de la variable objetivo:")
bal = df_vino['es_alta_calidad'].value_counts()
pct = df_vino['es_alta_calidad'].value_counts(normalize=True) * 100
print(f"   Alta calidad (quality ≥ 7) → clase 1: {bal[1]:,}  ({pct[1]:.1f}%)")
print(f"   Otra calidad (quality < 7) → clase 0: {bal[0]:,}  ({pct[0]:.1f}%)")
print(f"\nDistribución original de quality:")
print(df_vino['quality'].value_counts().sort_index())
df_vino.head()

In [ ]:
# ── CELDA 3: Validación y Exploración ────────────────────────────────────────
print("=" * 55)
print("VALIDACIÓN DEL DATASET")
print("=" * 55)

# Tipos y nulos
print("\n📋 Tipos de datos y valores nulos:")
print(df_vino.dtypes.to_frame('dtype').join(
    df_vino.isnull().sum().to_frame('nulos')
))

# Balance de clases
balance = df_vino['es_alta_calidad'].value_counts()
pct     = df_vino['es_alta_calidad'].value_counts(normalize=True) * 100
print(f"\n🎯 Balance de la variable objetivo:")
print(f"   Alta calidad  (1): {balance[1]:>5,}  ({pct[1]:.1f}%)")
print(f"   Otra calidad  (0): {balance[0]:>5,}  ({pct[0]:.1f}%)")

# Medias por clase
print(f"\n📊 Medias por clase (alta calidad vs resto):")
features_num = ['alcohol', 'volatile acidity', 'sulphates',
                'citric acid', 'fixed acidity', 'pH', 'density']
print(df_vino.groupby('es_alta_calidad')[features_num].mean().round(3).to_string())

## III. Preprocesamiento de Datos

### Pasos aplicados
1. **Verificación de nulos** — el dataset UCI no contiene valores faltantes.
2. **Escalado** — `StandardScaler` sobre todas las variables numéricas continuas.
3. **Definición de X e y** — todas las variables fisicoquímicas como predictores.

In [ ]:
# ── CELDA 4: Preprocesamiento ─────────────────────────────────────────────────
df = df_vino.copy()

# 4.1 Columnas a escalar — todas las fisicoquímicas
scale_cols = [
    'fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
    'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
    'pH', 'sulphates', 'alcohol'
]

# 4.2 Definir X e y
drop_cols = ['quality', 'es_alta_calidad']
X_raw = df.drop(columns=drop_cols)
y     = df['es_alta_calidad']

# 4.3 Escalar
scaler   = StandardScaler()
X_scaled = X_raw.copy()
X_scaled[scale_cols] = scaler.fit_transform(X_raw[scale_cols])

print("✅ Preprocesamiento completado")
print(f"   Shape X_raw    : {X_raw.shape}")
print(f"   Shape X_scaled : {X_scaled.shape}")
print(f"   Columnas finales: {X_scaled.columns.tolist()}")
print(f"\n   Clases en y → 0: {(y==0).sum():,}  |  1: {(y==1).sum():,}")
print("\n⚠️  StandardScaler aplicado — la Regresión Logística requiere escalado")
print("   para que los coeficientes sean comparables entre variables.")

## IV. Entrenamiento y Evaluación — Tres Splits

Para cada split se sigue el mismo proceso:
1. Dividir `X_scaled` e `y` en conjuntos de entrenamiento y prueba con `stratify=y`.
2. Entrenar `LogisticRegression` **solo** con los datos de entrenamiento.
3. Predecir **probabilidades** (`predict_proba`) y **clases** (`predict`) en el conjunto de prueba.
4. Calcular las 5 métricas requeridas: Accuracy, Precision, Recall, F1-Score, ROC-AUC.

In [ ]:
# ── CELDA 5: Función de entrenamiento + evaluación ───────────────────────────
def entrenar_evaluar(X, y, test_size, random_state=42, nombre="Split"):
    """
    Entrena LogisticRegression con un split train/test y devuelve
    un dict con métricas + datos para curva ROC y matriz de confusión.

    Nota: class_weight='balanced' compensa el desbalance ~14%/86%
    (217 clase-1 alta calidad vs 1382 clase-0 calidad estándar).
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    modelo = LogisticRegression(
        max_iter=1000,
        random_state=random_state,
        class_weight='balanced'   # compensa desbalance ~14%/86%
    )
    modelo.fit(X_train, y_train)

    y_pred = modelo.predict(X_test)
    y_prob = modelo.predict_proba(X_test)[:, 1]   # probabilidad clase 1

    fpr, tpr, _ = roc_curve(y_test, y_prob)

    metricas = {
        'nombre'    : nombre,
        'train_size': len(X_train),
        'test_size' : len(X_test),
        'accuracy'  : accuracy_score(y_test, y_pred),
        'precision' : precision_score(y_test, y_pred, zero_division=0),
        'recall'    : recall_score(y_test, y_pred, zero_division=0),
        'f1'        : f1_score(y_test, y_pred, zero_division=0),
        'roc_auc'   : roc_auc_score(y_test, y_prob),
        # Para curva ROC y matriz de confusión
        'fpr'       : fpr,
        'tpr'       : tpr,
        'y_test'    : y_test,
        'y_pred'    : y_pred,
        'modelo'    : modelo,
    }
    return metricas

print("✅ Función `entrenar_evaluar` definida correctamente")
print("   · LogisticRegression con class_weight='balanced' (desbalance ~14%/86%)")

In [ ]:
# ── CELDA 6: Ejecutar los tres splits ────────────────────────────────────────
resultados = [
    entrenar_evaluar(X_scaled, y, test_size=0.20, nombre="Split 80/20"),
    entrenar_evaluar(X_scaled, y, test_size=0.40, nombre="Split 60/40"),
    entrenar_evaluar(X_scaled, y, test_size=0.30, nombre="Split 70/30"),
]

# ── Imprimir métricas en consola ──────────────────────────────────────────────
print("=" * 65)
print(f"{'MÉTRICAS COMPARATIVAS — REGRESIÓN LOGÍSTICA BINARIA':^65}")
print("=" * 65)
header = f"{'Split':<12} {'N Train':>8} {'N Test':>8} {'Acc':>7} {'Prec':>7} {'Recall':>7} {'F1':>7} {'AUC':>7}"
print(header)
print("-" * 65)
for r in resultados:
    print(
        f"{r['nombre']:<12} {r['train_size']:>8,} {r['test_size']:>8,}"
        f" {r['accuracy']:>7.4f} {r['precision']:>7.4f}"
        f" {r['recall']:>7.4f} {r['f1']:>7.4f} {r['roc_auc']:>7.4f}"
    )
print("=" * 65)

## V. Visualizaciones

In [ ]:
# ── CELDA 7: Curvas ROC comparativas ─────────────────────────────────────────
COLORS = ['#1e3c72', '#27ae60', '#e67e22']

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot([0, 1], [0, 1], 'k--', lw=1.2, label='Clasificador aleatorio (AUC=0.50)')

for r, color in zip(resultados, COLORS):
    ax.plot(
        r['fpr'], r['tpr'], color=color, lw=2.2,
        label=f"{r['nombre']} — AUC = {r['roc_auc']:.4f}"
    )

ax.set_xlabel('Tasa de Falsos Positivos (FPR)', fontsize=12)
ax.set_ylabel('Tasa de Verdaderos Positivos (TPR)', fontsize=12)
ax.set_title('Curvas ROC — Regresión Logística Binaria\n'
             'Predicción de Vino de Alta Calidad (quality ≥ 7)', fontsize=13)
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.show()
print("✅ Curvas ROC generadas")

In [ ]:
# ── CELDA 8: Matrices de Confusión (3 splits) ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Matrices de Confusión — Regresión Logística Binaria — Wine Quality',
             fontsize=13, fontweight='bold')

for ax, r, color in zip(axes, resultados, COLORS):
    cm   = confusion_matrix(r['y_test'], r['y_pred'])
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['Calidad estándar (0)', 'Alta calidad (1)']
    )
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f"{r['nombre']}\nAccuracy={r['accuracy']:.4f}", fontsize=11)

plt.tight_layout()
plt.show()
print("✅ Matrices de confusión generadas")

In [ ]:
# ── CELDA 9: Gráfico comparativo de métricas ─────────────────────────────────
metricas_nombres = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
etiquetas        = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']

x     = np.arange(len(etiquetas))
width = 0.25

fig, ax = plt.subplots(figsize=(11, 5))
for i, (r, color) in enumerate(zip(resultados, COLORS)):
    vals = [r[m] for m in metricas_nombres]
    bars = ax.bar(x + i * width, vals, width, label=r['nombre'],
                  color=color, alpha=0.88)
    ax.bar_label(bars, fmt='%.3f', padding=2, fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels(etiquetas, fontsize=11)
ax.set_ylim([0, 1.12])
ax.set_ylabel('Valor de la Métrica', fontsize=11)
ax.set_title('Comparación de Métricas — Tres Splits\n'
             'Regresión Logística Binaria — Wine Quality',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.axhline(y=1.0, color='#ccc', linewidth=0.8, linestyle='--')
plt.tight_layout()
plt.show()
print("✅ Gráfico de métricas comparativas generado")

In [ ]:
# ── CELDA 10: Importancia de coeficientes (Split 80/20) ──────────────────────
modelo_ref = resultados[0]['modelo']
coefs      = pd.Series(modelo_ref.coef_[0], index=X_scaled.columns)
coefs_ord  = coefs.abs().sort_values(ascending=True)

# Gráfico de coeficientes absolutos
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Coeficientes — Regresión Logística (Split 80/20) — Wine Quality',
             fontsize=13, fontweight='bold')

# Panel izquierdo: coeficientes absolutos (importancia)
coefs_ord.plot(kind='barh', ax=axes[0], color='#2a5298', alpha=0.85)
axes[0].set_xlabel('|Coeficiente| — Importancia relativa', fontsize=11)
axes[0].set_title('Importancia (valor absoluto)', fontweight='bold')
axes[0].axvline(x=0, color='black', linewidth=0.8)
for patch in axes[0].patches:
    w = patch.get_width()
    if w > 0.01:
        axes[0].text(w + 0.01, patch.get_y() + patch.get_height() / 2,
                     f'{w:.3f}', va='center', fontsize=9)

# Panel derecho: coeficientes con signo (dirección)
coefs_signed = coefs.sort_values()
colors_sign  = ['#e74c3c' if v > 0 else '#3498db' for v in coefs_signed]
coefs_signed.plot(kind='barh', ax=axes[1], color=colors_sign, alpha=0.85)
axes[1].set_xlabel('Coeficiente (con signo)', fontsize=11)
axes[1].set_title('Dirección del efecto\n🔴 positivo → aumenta P(alta calidad)\n🔵 negativo → reduce P(alta calidad)',
                  fontweight='bold', fontsize=10)
axes[1].axvline(x=0, color='black', linewidth=1.5)

plt.tight_layout()
plt.show()

print("✅ Coeficientes generados")
print(f"\n📊 Top 5 variables con mayor influencia positiva (aumentan P(alta calidad)):")
print(coefs.sort_values(ascending=False).head(5).to_string())
print(f"\n📊 Top 5 variables con mayor influencia negativa (reducen P(alta calidad)):")
print(coefs.sort_values(ascending=True).head(5).to_string())
print(f"\n📌 Comparación con correlación Spearman (EDA notebook V1):")
print(f"   alcohol          → correlación +0.48  — se espera coeficiente positivo")
print(f"   volatile acidity → correlación -0.39  — se espera coeficiente negativo")
print(f"   sulphates        → correlación +0.37  — se espera coeficiente positivo")

## VI. Tabla Final de Métricas y Conclusiones

In [ ]:
# ── CELDA 11: DataFrame resumen de métricas ───────────────────────────────────
resumen = pd.DataFrame([{
    'Split'          : r['nombre'],
    'N Entrenamiento': r['train_size'],
    'N Prueba'       : r['test_size'],
    'Accuracy'       : round(r['accuracy'],  4),
    'Precision'      : round(r['precision'], 4),
    'Recall'         : round(r['recall'],    4),
    'F1-Score'       : round(r['f1'],        4),
    'ROC-AUC'        : round(r['roc_auc'],   4),
} for r in resultados])

resumen.set_index('Split', inplace=True)

print("=" * 65)
print(f"{'TABLA FINAL DE MÉTRICAS':^65}")
print("=" * 65)
try:
    display(resumen.style
        .background_gradient(cmap='Greens',
                             subset=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'])
        .format('{:.4f}',
                subset=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'])
        .set_caption("Regresión Logística Binaria — Predicción de Vino de Alta Calidad (quality ≥ 7)")
    )
except:
    print(resumen.to_string())

## VII. Conclusiones

### ¿Qué aprendimos?

| Aspecto | Observación |
|---------|------------|
| **Estabilidad del modelo** | Si los tres splits arrojan métricas similares, el modelo generaliza bien y no depende del azar de una partición específica. |
| **Accuracy vs AUC** | El Accuracy puede ser engañoso dado el desbalance ~14%/86%. El ROC-AUC es la métrica más robusta para evaluar este modelo. |
| **Precision vs Recall** | Existe un *trade-off*: aumentar el umbral de decisión sube la Precision (menos falsos positivos de alta calidad) pero baja el Recall (se pierden vinos realmente buenos). |
| **Coeficientes** | `alcohol` (positivo) y `volatile acidity` (negativo) dominan la predicción, consistente con el EDA y la correlación Spearman del notebook V1. |
| **Desbalance de clases** | Solo ~14% de los vinos son de alta calidad. `class_weight='balanced'` es crítico para que el modelo no ignore esta clase minoritaria. |
| **Limitación del modelo** | La Regresión Logística asume una relación lineal entre las variables y el log-odds. Las interacciones no lineales entre propiedades químicas del vino requieren modelos más complejos. |

### Comparación con el Árbol de Decisión CART

| | Regresión Logística | Árbol CART |
|---|---|---|
| **Escalado** | Requerido (StandardScaler) | No requerido |
| **Frontera de decisión** | Lineal | No lineal (por partes) |
| **Importancia de variables** | Coeficientes (log-odds) | Reducción impureza Gini |
| **Riesgo de overfitting** | Bajo (regularización implícita) | Alto si max_depth no se controla |
| **Interpretabilidad** | Ecuación logística | Árbol de reglas if-else |

### Próximos pasos sugeridos
- Ajustar el umbral de clasificación (por defecto 0.5) según la prioridad: si importa más no perder vinos buenos → bajar umbral (mayor Recall); si importa no etiquetar vinos mediocres como buenos → subir umbral (mayor Precision).
- Evaluar si cambiar el umbral de binarización (actualmente quality ≥ 7) a quality ≥ 6 mejora el balance de clases.
- Explorar modelos más complejos: `RandomForestClassifier`, `GradientBoostingClassifier`.
- Aplicar validación cruzada (k-fold) para una estimación más robusta de las métricas.